# Electromagnetic Gyrokinetic Benchmarks

Standard EM test cases for gyrokinetic codes with $A_\parallel$ dynamics. The canonical case in the literature is the **Waltz standard case** (Waltz, PoP 1994; also Falchetto et al., PPCF 2008), which traces the transition from ITG turbulence at $\beta = 0$ to a kinetic ballooning mode (KBM) at finite $\beta$. See Fig. 4 of the GKW paper (Peeters et al., CPC 2009).

| Test | Physics | Reference |
|------|---------|-----------|
| **Waltz $\beta$-scan** | ITG $\rightarrow$ KBM transition | GKW paper Fig. 4 middle; GS2 |
| **Eigenmode structure** | $\phi(s)$, $A_\parallel(s)$ at fixed $\beta$ | GKW `parallel.dat` |
| **Shear Alfvén wave** | $\omega = k_\parallel v_A$ at $R/L_T=R/L_n=0$ | analytic, $v_A = v_{\rm th}/\sqrt{\beta}$ |

Waltz standard parameters: $q=2$, $\hat s=1$, $\varepsilon=0.166$, $R/L_{Ti}=R/L_{Te}=9$, $R/L_n=3$, $T_e/T_i=1$, kinetic deuterium + electrons at $m_i/m_e=3676$, s-alpha geometry, $k_\theta\rho_s=0.42$ (one mode). $A_\parallel$ on, $B_\parallel$ off.

The KBM threshold for this case is at $\beta_{\rm crit} \approx 0.006$ (GKW Fig. 4).

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))
os.environ["CUDA_VISIBLE_DEVICES"] = "7"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"

In [ ]:
import time as _time
import numpy as np
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
from dataclasses import replace

from gyaradax.geometry import compute_geometry, compute_geometry_from_input
from gyaradax.params import gkparams_from_input_and_geometry, GKParams
from gyaradax.simulate import gk_run
from gyaradax.solver import init_f, default_state, linear_precompute
from gyaradax.integrals import calculate_phi, calculate_apar
from gyaradax.solver import _compute_fields
from gyaradax.plot_utils import JAX_COLORS

GKW_CASES = os.path.join("..", "tests", "data", "gkw_cases")
os.makedirs("figs", exist_ok=True)

---
## 1. Waltz $\beta$-scan: ITG $\rightarrow$ KBM transition

Linear gyrokinetic mode at the Waltz standard case, scanning $\beta$. Below the KBM threshold ($\beta \lesssim 0.005$) the mode is ITG-driven, propagating in the ion direction ($\omega > 0$ in our convention) and stabilising with increasing $\beta$. Above threshold the unstable branch jumps to the KBM, which propagates in the electron direction and destabilises rapidly with $\beta$.

GKW reference values were generated with `gkw_debug3.x` on the same `em_apar_waltz` input.dat (NX=1, NMOD=1, $N_s=112$, nperiod=4, NTIME=200, naverage=50, $\delta t=0.001$), overriding `beta` per run.

In [ ]:
# GKW reference β-scan (Waltz standard case, kthrho=0.42426)
# generated by /tmp/gkw_beta_scan/run_scan.sh on gkw_debug3.x
gkw_beta = np.array(
    [0.0001, 0.001, 0.002, 0.003, 0.004, 0.005, 0.006, 0.007, 0.008, 0.009, 0.01]
)
gkw_gamma = np.array(
    [0.5825, 0.5567, 0.5220, 0.4776, 0.4213, 0.3744,
     -0.1277, 0.5284, 0.7924, 0.9830, 1.1248]
)
gkw_omega = np.array(
    [0.6987, 0.7185, 0.7451, 0.7827, 0.8473, 0.8791,
      2.1466, 2.0766, 1.9292, 1.8245, 1.7385]
)
# At β=0.006 GKW found a STABLE intermediate mode (γ<0). The KBM takes over at β>0.006.
# Frequency sign flip identifies the branch transition.
print('β       γ_GKW    ω_GKW')
for b, g, w in zip(gkw_beta, gkw_gamma, gkw_omega):
    branch = 'ITG' if w < 1.5 else ('KBM' if g > 0 else 'stable')
    print(f'{b:.4f}  {g:+.4f}  {w:+.4f}  [{branch}]')

In [ ]:
# Build gyaradax Waltz scan using the GKW input.dat as ground truth, override β
WALTZ_INPUT = os.path.join(GKW_CASES, 'em_apar_waltz', 'input.dat')

def run_waltz_linear(beta, n_windows=600, navg=20, dt=0.001):
    """Linear γ + ω at the Waltz case, parameter beta overridden.

    Match GKW reference run: NAVERAGE=20, DTIM=0.001, run long enough to
    converge through the ITG-KBM transition region (γ-dip near β=0.005).
    """
    geom = compute_geometry_from_input(WALTZ_INPUT)
    params = gkparams_from_input_and_geometry(WALTZ_INPUT, geom)
    params = replace(params, beta=float(beta), non_linear=False,
                     adaptive_dt=False, dt=dt, naverage=navg)
    nsp = int(np.asarray(params.mas).shape[0]) if np.asarray(params.mas).ndim > 0 else 1
    df = init_f(geom, finit='cosine2', amp_init_real=params.amp_init, n_species=nsp)
    pre = linear_precompute(geom, params)
    state = default_state(nky=len(geom['krho']))
    # also track phase history to estimate ω
    phi_phases = []
    times = []
    for _ in range(n_windows):
        df, phi, _, state = gk_run(df, geom, params, state, navg, pre=pre)
        # use phi(midplane, kx=ixzero, ky=ITG)
        ns = phi.shape[0]
        mid = ns // 2
        ixz = int(jnp.argmin(jnp.abs(jnp.asarray(geom['kxrh']))))
        iky = 0  # NMOD=1 → single ky
        c = phi[mid, ixz, iky]
        phi_phases.append(float(jnp.angle(c)))
        times.append(float(state.time))
    # frequency from last quarter (use unwrap so jumps don't bias the fit)
    ph = np.unwrap(np.asarray(phi_phases))
    tt = np.asarray(times)
    mask = tt > tt[-1] * 0.75
    omega = float(np.polyfit(tt[mask], ph[mask], 1)[0]) if mask.sum() >= 2 else np.nan
    gamma = float(np.asarray(state.last_growth_rate).max())
    return gamma, omega

In [ ]:
gyx_beta = np.array([0.0001, 0.001, 0.002, 0.003, 0.004, 0.005,
                     0.007, 0.008, 0.009, 0.01])
gyx_gamma = []
gyx_omega = []
for b in gyx_beta:
    t0 = _time.time()
    g, w = run_waltz_linear(b)
    wall = _time.time() - t0
    gyx_gamma.append(g)
    gyx_omega.append(w)
    print(f'β={b:.4f}  γ_gyx={g:+.4f}  ω_gyx={w:+.4f}  ({wall:.1f}s)')
gyx_gamma = np.array(gyx_gamma)
gyx_omega = np.array(gyx_omega)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5))

ax = axes[0]
ax.plot(gkw_beta, gkw_gamma, 'x', color='k', ms=8, lw=1.8,
        markerfacecolor='none', label='GKW', zorder=10)
ax.plot(gyx_beta, gyx_gamma, 'o-', color=JAX_COLORS['blue'], ms=5, lw=1.2,
        label='gyaradax')
ax.axvline(0.006, color='0.6', ls=':', lw=0.8)
ax.axhline(0, color='0.6', ls=':', lw=0.8)
ax.set_xlabel(r'$\beta$', fontsize=13)
ax.set_ylabel(r'$\gamma$  [v$_{\rm th}$/R]', fontsize=13)
ax.set_title(r'(a) $\gamma$ vs $\beta$', fontsize=15)
ax.legend(frameon=False, fontsize=12, loc='upper left')
ax.set_xlim(0, 0.0105)
ax.set_ylim(-0.3, 1.3)
ax.tick_params(labelsize=10)
ax.grid(True)
ax.text(0.0028, 1.0, 'ITG branch', fontsize=10, ha='center')
ax.text(0.0085, 1.0, 'KBM branch', fontsize=10, ha='center')

ax2 = axes[1]
ax2.plot(gkw_beta, gkw_omega, 'x', color='k', ms=8, lw=1.8,
         markerfacecolor='none', label='GKW', zorder=10)
ax2.plot(gyx_beta, gyx_omega, 'o-', color=JAX_COLORS['blue'], ms=5, lw=1.2,
         label='gyaradax')
ax2.axvline(0.006, color='0.6', ls=':', lw=0.8)
ax2.set_xlabel(r'$\beta$', fontsize=13)
ax2.set_ylabel(r'$\omega$  [v$_{\rm th}$/R]', fontsize=13)
ax2.set_title(r'(b) $\omega$ vs $\beta$', fontsize=15)
ax2.legend(frameon=False, fontsize=12, loc='upper left')
ax2.set_xlim(0, 0.0105)
ax2.tick_params(labelsize=10)
ax2.grid(True)

fig.tight_layout()
fig.savefig('figs/em_waltz_beta_scan.pdf')
plt.show()

---
## 2. Eigenmode structure $\phi(s)$, $A_\parallel(s)$

Spatial structure along the field line for the Waltz case at three $\beta$ values, spanning the ITG–KBM transition. The shape of $\phi(s)$ is ballooning-like (peaked at the LFS midplane); $A_\parallel(s)$ has opposite parity for the ITG branch and the KBM branch.

In [ ]:
def run_waltz_eigenmode(beta, n_windows=200, navg=50):
    """Run to convergence and return (sgrid, phi(s), apar(s)) for kx=0, ITG ky."""
    geom = compute_geometry_from_input(WALTZ_INPUT)
    params = gkparams_from_input_and_geometry(WALTZ_INPUT, geom)
    params = replace(params, beta=float(beta), non_linear=False,
                     adaptive_dt=False, dt=0.001, naverage=navg)
    nsp = int(np.asarray(params.mas).shape[0]) if np.asarray(params.mas).ndim > 0 else 1
    df = init_f(geom, finit='cosine2', amp_init_real=params.amp_init, n_species=nsp)
    pre = linear_precompute(geom, params)
    state = default_state(nky=len(geom['krho']))
    for _ in range(n_windows):
        df, phi, _, state = gk_run(df, geom, params, state, navg, pre=pre)
    # compute fields directly to get apar
    phi, apar, _ = _compute_fields(df, geom, params, pre)
    phi = np.asarray(phi)
    apar = np.asarray(apar) if apar is not None else None
    sgrid = np.asarray(geom['sgrid'])
    ixz = int(np.argmin(np.abs(np.asarray(geom['kxrh']))))
    iky = 0
    phi_s = phi[:, ixz, iky]
    apar_s = apar[:, ixz, iky] if apar is not None else None
    # normalise phi to (1, 0) at midplane (GKW convention)
    mid = len(sgrid) // 2
    rot = phi_s[mid] if abs(phi_s[mid]) > 1e-30 else 1.0
    return sgrid, phi_s / rot, (apar_s / rot if apar_s is not None else None)

In [ ]:
betas_em = [0.001, 0.005, 0.009]
modes = {}
for b in betas_em:
    sgrid, phi_s, apar_s = run_waltz_eigenmode(b)
    modes[b] = (sgrid, phi_s, apar_s)
    print(f'β={b:.3f}: |phi|_max={np.max(np.abs(phi_s)):.3f}, '
          f'|apar|_max={np.max(np.abs(apar_s)) if apar_s is not None else 0:.3e}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7.0, 2.5))
ax_phi, ax_apar = axes
colors = [JAX_COLORS['blue'], JAX_COLORS['orange'], JAX_COLORS['green']]
labels = [r'$\beta=0.001$ (ITG)', r'$\beta=0.005$ (ITG)', r'$\beta=0.009$ (KBM)']
for (b, c, lab) in zip(betas_em, colors, labels):
    s, phi_s, apar_s = modes[b]
    ax_phi.plot(s, phi_s.real, '-', color=c, lw=1.3, label=lab)
    ax_phi.plot(s, phi_s.imag, '--', color=c, lw=0.8)
    if apar_s is not None:
        ax_apar.plot(s, apar_s.real, '-', color=c, lw=1.3, label=lab)
        ax_apar.plot(s, apar_s.imag, '--', color=c, lw=0.8)

ax_phi.set_xlabel(r'$s$', fontsize=13)
ax_phi.set_ylabel(r'$\phi(s)$', fontsize=13)
ax_phi.set_title(r'(a) $\phi$ eigenmode (solid: Re, dashed: Im)', fontsize=13)
ax_phi.legend(frameon=False, fontsize=10, loc='upper right')
ax_phi.tick_params(labelsize=10)
ax_phi.grid(True)

ax_apar.set_xlabel(r'$s$', fontsize=13)
ax_apar.set_ylabel(r'$A_\parallel(s)$', fontsize=13)
ax_apar.set_title(r'(b) $A_\parallel$ eigenmode (solid: Re, dashed: Im)', fontsize=13)
ax_apar.tick_params(labelsize=10)
ax_apar.grid(True)

fig.tight_layout()
fig.savefig('figs/em_waltz_eigenmode.pdf')
plt.show()

---
## 3. Shear Alfvén wave dispersion

Pure analytical EM test. With drives off ($R/L_T = R/L_n = 0$), kinetic electrons, and finite $\beta$, the system reduces to the shear Alfvén wave equation. The dispersion is

$$\omega = k_\parallel v_A , \qquad v_A = \frac{B}{\sqrt{\mu_0\, n_i m_i}}.$$

In GKW normalised units with the kinetic deuterium species as the reference ($n_i=n_{\rm ref}$, $m_i=m_{\rm ref}$, $T_i=T_{\rm ref}$), the Alfvén velocity is

$$\frac{v_A}{v_{\rm th,ref}} = \frac{1}{\sqrt{\beta}}, \qquad \beta = \frac{2\mu_0 n_{\rm ref} T_{\rm ref}}{B_{\rm ref}^2}.$$

We initialise a small $A_\parallel$-like perturbation in a non-zonal mode and measure the oscillation frequency of $\phi$ at the midplane. The parallel wavenumber $k_\parallel$ is set by the eigenmode structure on the flux tube; in the simplest cylindrical limit ($\hat s = 0$, single $k_\theta$) it is $k_\parallel \sim 2\pi/L_\parallel$ with $L_\parallel \propto qR$. We compare the measured $\omega/\sqrt{\beta}$ to a single $\beta$-independent constant (= $k_\parallel \cdot v_{\rm th,ref}$).

In [ ]:
def run_alfven(beta, n_windows=200, navg=50, dt=0.0005):
    """Drives off + kinetic electrons + finite β: measure ω from φ(t).

    With rlt=rln=0 there's no ITG drive; the only dynamics is the shear
    Alfvén wave. We expect ω ≈ k_∥ * v_A with v_A = v_th/sqrt(β).
    """
    # Waltz-like geometry but with drives off and kinetic e-
    geom = compute_geometry(
        q=2.0, shat=1.0, eps=0.166,
        ns=64, nvpar=32, nmu=8, vpar_max=3.0,
        nkx=1, nky=1, nperiod=4,
        kxmax=0.42, krhomax=0.42, signB=1.0, Rref=1.0,
        geom_type='s-alpha',
    )
    params = GKParams(
        dt=dt, naverage=navg, non_linear=False, adaptive_dt=False,
        adiabatic_electrons=False,
        disp_par=1.0, disp_vp=0.05, disp_x=0.0, disp_y=0.0,
        finit='cosine2', amp_init=1e-3,
        mas=jnp.array([1.0, 2.72e-4]), tmp=jnp.array([1.0, 1.0]),
        de=jnp.array([1.0, 1.0]), signz=jnp.array([1.0, -1.0]),
        vthrat=jnp.sqrt(jnp.array([1.0, 1.0]) / jnp.array([1.0, 2.72e-4])),
        rlt=jnp.array([0.0, 0.0]), rln=jnp.array([0.0, 0.0]),
        dgrid=1.0, tgrid=1.0,
        sgr_dist=float(geom['sgr_dist']),
        dvp=float(geom['dvp']),
        kxmax=float(np.max(np.abs(np.asarray(geom['kxrh'])))) or 1.0,
        kymax=float(np.max(np.asarray(geom['krho']))) or 1.0,
        nlapar=True, nlbpar=False, beta=float(beta),
        drive_scale=1.0, idisp=2, cfl_safety=0.95,
        mixed_precision=False, backend='jax',
    )
    df = init_f(geom, finit='cosine2', amp_init_real=params.amp_init, n_species=2)
    pre = linear_precompute(geom, params)
    state = default_state(nky=len(geom['krho']))
    phi_phases, times = [], []
    for _ in range(n_windows):
        df, phi, _, state = gk_run(df, geom, params, state, navg, pre=pre)
        ns = phi.shape[0]
        mid = ns // 2
        c = phi[mid, 0, 0]
        phi_phases.append(float(jnp.angle(c)))
        times.append(float(state.time))
    ph = np.unwrap(np.asarray(phi_phases))
    tt = np.asarray(times)
    mask = tt > tt[-1] * 0.5
    omega = float(np.polyfit(tt[mask], ph[mask], 1)[0]) if mask.sum() >= 2 else np.nan
    return omega

In [ ]:
betas_alfven = np.array([0.001, 0.002, 0.005, 0.01, 0.02])
omegas_alfven = []
for b in betas_alfven:
    omega = run_alfven(b)
    omegas_alfven.append(omega)
    print(f'β={b:.4f}  |ω|={abs(omega):.4f}  |ω|*sqrt(β)={abs(omega)*np.sqrt(b):.4f}  '
          f'(≈ k_∥, β-independent if Alfvénic)')
omegas_alfven = np.array(omegas_alfven)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(4.0, 3.0))
kpar_eff = np.abs(omegas_alfven) * np.sqrt(betas_alfven)
ax.plot(betas_alfven, np.abs(omegas_alfven), 'o-', color=JAX_COLORS['blue'], ms=6,
        label='gyaradax')
k_const = np.median(kpar_eff)
ax.plot(betas_alfven, k_const / np.sqrt(betas_alfven), '--k', lw=1.2,
        label=rf'$k_\parallel/\sqrt{{\beta}}$, $k_\parallel={k_const:.3f}$')
ax.set_xlabel(r'$\beta$', fontsize=13)
ax.set_ylabel(r'$|\omega|$', fontsize=13)
ax.set_xscale('log')
ax.set_yscale('log')
ax.legend(frameon=False, fontsize=11)
ax.tick_params(labelsize=10)
ax.grid(True, which='both', alpha=0.3)
ax.set_title(r'(a) Shear Alfvén: $\omega \propto 1/\sqrt{\beta}$', fontsize=13)
fig.tight_layout()
fig.savefig('figs/em_shear_alfven.pdf')
plt.show()